# 配套实践 10-01：构造多模态 token、时间与 mask

本练习不训练网络，而是完成进入 Context Transformer 之前最容易出错的数据整理：把不同频率传感器对齐到决策时刻，为 token 写入模态与时间身份，并区分 padding 和相机掉帧。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/10-multimodal-context-model/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 处理传感器时间戳、索引和有效位
import matplotlib.pyplot as plt  # 绘制时间对齐、token 布局和 mask
from matplotlib.patches import Rectangle  # 绘制不同模态的彩色 token 方块
np.random.seed(101)  # 固定示例中的随机测量值
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 把多速率测量对齐到策略决策时刻

下面模拟相机、本体和触觉的采样时间。对每个决策时刻，相机只选择最近的过去帧，本体使用最近值，触觉则统计上一个控制窗口内的峰值。这样不会借用未来数据，也不会把短暂接触峰值平均掉。

In [ ]:
decision_times = np.array([0.10, 0.20, 0.30, 0.40])  # 定义策略产生动作的四个决策时刻
camera_times = np.array([0.02, 0.07, 0.13, 0.19, 0.26, 0.34, 0.39])  # 定义低频且略有抖动的相机时间戳
proprio_times = np.arange(0.00, 0.401, 0.02)  # 定义较高频率的本体状态时间戳
tactile_times = np.arange(0.00, 0.401, 0.01)  # 定义更高频率的触觉时间戳
tactile_values = 0.05 * np.random.rand(len(tactile_times))  # 生成低幅背景触觉读数
tactile_values[(tactile_times >= 0.23) & (tactile_times <= 0.25)] += 0.9  # 在短时间窗口中加入一次接触峰值
def latest_past_index(sample_times, decision_time):  # 定义只选择当前或过去测量的函数
    valid_indices = np.where(sample_times <= decision_time)[0]  # 找到所有不晚于决策时刻的索引
    return int(valid_indices[-1])  # 返回时间上最接近决策时刻的过去测量
camera_indices = [latest_past_index(camera_times, time) for time in decision_times]  # 为每个决策时刻选择相机帧
proprio_indices = [latest_past_index(proprio_times, time) for time in decision_times]  # 为每个决策时刻选择本体状态
window_edges = np.concatenate([[0.0], decision_times])  # 建立触觉聚合窗口的左右边界
tactile_peaks = []  # 准备保存每个控制窗口的触觉峰值
for window_index in range(len(decision_times)):  # 逐个决策窗口统计触觉信号
    in_window = (tactile_times > window_edges[window_index]) & (tactile_times <= window_edges[window_index + 1])  # 找到当前窗口内的触觉采样
    tactile_peaks.append(float(tactile_values[in_window].max()))  # 保存窗口最大值避免遗漏短暂接触
fig, axes = plt.subplots(3, 1, figsize=(10, 5.7), sharex=True)  # 创建相机、本体和触觉三条时间轴
axes[0].scatter(camera_times, np.ones_like(camera_times), color="#2563eb", label="Camera samples")  # 显示相机实际采样时刻
axes[0].scatter(camera_times[camera_indices], np.ones(len(camera_indices)), s=100, facecolors="none", edgecolors="#ea580c", label="Selected")  # 圈出各决策使用的过去帧
axes[1].scatter(proprio_times, np.ones_like(proprio_times), color="#16a34a", s=18, label="Proprio samples")  # 显示高频本体状态采样
axes[1].scatter(proprio_times[proprio_indices], np.ones(len(proprio_indices)), s=90, facecolors="none", edgecolors="#ea580c", label="Selected")  # 圈出各决策使用的本体值
axes[2].plot(tactile_times, tactile_values, color="#7c3aed", label="Tactile signal")  # 绘制包含短暂峰值的触觉曲线
axes[2].scatter(decision_times, tactile_peaks, color="#ea580c", zorder=3, label="Window peak")  # 在决策时刻显示各窗口聚合峰值
for axis in axes:  # 为三条时间轴统一加入决策时刻标记
    for decision_time in decision_times:  # 遍历四个策略决策时刻
        axis.axvline(decision_time, color="#94a3b8", linestyle="--", linewidth=1)  # 用虚线显示同一决策时间
    axis.legend(loc="upper left", ncol=2)  # 显示当前传感器的图例
    axis.set_yticks([])  # 隐藏不承载数值意义的纵轴刻度
axes[2].set(xlabel="Timestamp (s)", ylabel="Touch")  # 标注真实时间与触觉幅值
fig.suptitle("Align every modality without reading the future")  # 强调对齐必须遵守因果时间
fig.tight_layout()  # 调整三幅子图间距
plt.show()  # 显示多速率传感器对齐结果

**怎样理解结果：** 灰色虚线是动作决策时刻，橙色空心圈标出被选择的相机帧和本体状态。被选择点都位于虚线左侧或恰好重合，因此没有未来泄漏。触觉在 0.23～0.25 秒出现短峰，第三个窗口的橙色点仍能保留它；若只在 0.30 秒抽取单点，接触事件可能已经消失。

## 2. 从对齐结果建立 token 布局

我们使用语言 token 加两个历史时刻。每个时刻依次放置两台相机、本体、触觉和上一动作。第二台相机在当前时刻掉帧，因此该位置仍保留来源身份，但有效位为 0。

In [ ]:
token_types = ["Language", "Cam1", "Cam2", "State", "Touch", "Prev action", "Cam1", "Cam2", "State", "Touch", "Prev action"]  # 按固定 schema 列出全部 token 类型
token_times = ["global", "t-1", "t-1", "t-1", "t-1", "t-1", "t", "t", "t", "t", "t"]  # 为每个 token 标记全局或历史时间
validity = np.array([1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1], dtype=bool)  # 把当前时刻第二台相机标记为掉帧
type_colors = {"Language": "#fed7aa", "Cam1": "#bfdbfe", "Cam2": "#bfdbfe", "State": "#bbf7d0", "Touch": "#ddd6fe", "Prev action": "#fecaca"}  # 为不同模态分配稳定颜色
fig, axis = plt.subplots(figsize=(11, 3.3))  # 创建 token 序列画布
for token_index, (token_type, token_time, is_valid) in enumerate(zip(token_types, token_times, validity)):  # 逐个绘制 token 方块
    face_color = type_colors[token_type] if is_valid else "#e2e8f0"  # 使用灰色表示本次缺失的位置
    edge_style = "-" if is_valid else "--"  # 使用虚线边框进一步强调缺失 token
    axis.add_patch(Rectangle((token_index, 0), 0.9, 1.0, facecolor=face_color, edgecolor="#475569", linestyle=edge_style, linewidth=1.5))  # 绘制当前 token 的边界和填充
    axis.text(token_index + 0.45, 0.62, token_type.replace("Prev action", "Action"), ha="center", va="center", fontsize=9)  # 在方块中写入模态名称
    axis.text(token_index + 0.45, 0.27, token_time, ha="center", va="center", fontsize=9, color="#475569")  # 在方块中写入时间身份
    axis.text(token_index + 0.45, -0.24, f"valid={int(is_valid)}", ha="center", va="center", fontsize=8)  # 在方块下写出有效位
axis.set(xlim=(-0.2, len(token_types)), ylim=(-0.55, 1.35), title="Token identity = content + modality + time + source")  # 设置序列范围并强调 token 身份组成
axis.axis("off")  # 隐藏与 token 序列无关的坐标轴
fig.tight_layout()  # 压缩图像空白边距
plt.show()  # 显示统一多模态 token 布局

**怎样理解结果：** 两个 Cam2 方块具有相同相机身份和不同时间身份；灰色虚线方块表示当前 Cam2 位置存在于 schema 中，但本次没有有效图像。共同的向量维度不会抹去这些身份，因为模型还会加入模态、时间和相机 embedding。有效位则告诉 Attention 这个缺失位置不可作为 Key 和 Value。

## 3. 区分掉帧 mask 与 batch padding

现在把上述 11-token 样本与一个只有单步历史的短样本放进同一 batch。长样本的 Cam2 掉帧属于模态缺失；短样本末尾补齐出来的位置属于 padding。它们都会禁止读取，但原因和后续统计不能混为一谈。

In [ ]:
long_sample_valid = validity.copy()  # 复制包含一次相机掉帧的长样本有效位
short_sample_valid = np.array([1, 1, 1, 1, 1, 1] + [0] * 5, dtype=bool)  # 为短样本把不存在的后五个位置标记为 padding
attention_key_mask = np.stack([long_sample_valid, short_sample_valid])  # 组成 batch 中每个样本可作为 Key 的位置
missing_modality_mask = np.stack([~validity, np.zeros_like(validity)])  # 单独记录真实时间位置上的模态掉帧
padding_mask = np.stack([np.zeros_like(validity), np.array([0] * 6 + [1] * 5, dtype=bool)])  # 单独记录为对齐 batch 长度而补出的 token
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))  # 创建最终可读位置、掉帧和 padding 三幅矩阵
matrix_specs = [(attention_key_mask, "Readable keys"), (missing_modality_mask, "Missing modality"), (padding_mask, "Batch padding")]  # 组织三种有效性矩阵及标题
for axis, (matrix, title) in zip(axes, matrix_specs):  # 依次绘制三种 mask
    axis.imshow(matrix.astype(int), cmap="Blues", vmin=0, vmax=1, aspect="auto")  # 用深浅颜色显示布尔值
    axis.set(title=title, xlabel="Token position", ylabel="Sample", xticks=range(len(token_types)), yticks=[0, 1])  # 标注样本行和 token 列
    for row in range(matrix.shape[0]):  # 遍历两个 batch 样本
        for column in range(matrix.shape[1]):  # 遍历十一个 token 位置
            axis.text(column, row, str(int(matrix[row, column])), ha="center", va="center", fontsize=8)  # 在单元格中写出零或一
fig.suptitle("The same blocked position can have different causes")  # 强调禁止读取的来源需要分开记录
fig.tight_layout()  # 调整三幅矩阵的间距
plt.show()  # 显示多模态有效性与 padding 的区别

**怎样理解结果：** 第一幅图直接用于 Attention：值为 1 的列可以被读取。第二幅图只在长样本的掉帧相机位置为 1，说明真实时刻存在但该模态失效；第三幅图只标记短样本为凑齐 batch 而增加的尾部位置。训练和部署监控应保留后两种原因，而不是只留下一个合并 mask。

**本练习的结论：** Context Model 的输入质量首先取决于时间戳、schema 和有效位。Transformer 无法替数据管线猜出一个零向量代表真实零值、相机掉帧还是 batch 补齐。